In [8]:
#Final Project CS 437
#Shaheen Nafeie, Navid Nafeie, Jason Pham

import pandas as pd 
import sklearn
from sklearn.ensemble import GradientBoostingRegressor

# Load dataset into a Pandas DataFrame
my_data = pd.read_csv("train10k.csv")

# Fill missing values in the dataset with with 0
my_data.fillna(value=0, inplace=True)

# Count total number of rows in the dataset
total_rows = len(my_data)

# Output the total number of rows
print(total_rows)

# Display column names in the dataset
my_data.columns

# Select a subset of data containing specific columns
lin_data = my_data[['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP', 'Target']]

# Split the data into training (70%) and testing (30%) 
my_train= lin_data.sample(frac =0.7)

# Use remaining data for testing
my_test= lin_data.drop(my_train.index)

# Separate the features (independent varibles) and target (dependent variable)
train_x = my_train.drop( columns = [ 'Target' ] )

# Target values for training
train_y = my_train[ 'Target' ]


# Separate the features for testing
test_x=my_test.drop(columns = ['Target'])

# Separate the target values for testing
test_y = my_test[ ['Target'] ]



10000


In [11]:
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import TransformedTargetRegressor
import numpy as np

from sklearn.model_selection import GridSearchCV

# Custom transformer to select specific columns
class SelectColumns( BaseEstimator, TransformerMixin ):

    def __init__( self, columns ):
        self.columns = columns

    def fit( self, xs, ys, **params ):
        return self

    def transform( self, xs ):
        return xs[ self.columns ]


# Standard scaler for feature scaling
scalar=StandardScaler()

# Define pipeline stages
stages = [
( 'column_select', SelectColumns( ['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP'] ) ),
('scalar', StandardScaler()),
 ( 'Gradient_Boosting_Regressor', GradientBoostingRegressor())
]

# Create a pipeline 
my_pipe=Pipeline(stages)

# Split the data into features (xs) and target (ys)
xs = lin_data.drop( columns = [ 'Target' ] )
ys = lin_data[ 'Target' ]


# Split the data into training and testing sets (70% for training and 30% testing)
train_x, test_x, train_y, test_y = train_test_split( xs, ys, train_size = 0.7 )

# Fit the pipeline on the training data
my_pipe.fit( train_x, train_y )


# Define a hyperparaameter grid for tuning the gradient boosting regressor
hyperparam_grid = {
            'Gradient_Boosting_Regressor__n_estimators':[100],
            'Gradient_Boosting_Regressor__learning_rate':[0.01],
            'Gradient_Boosting_Regressor__max_depth':[10,20],

}    

# Initialize GridSearchCV with the pipeline, hyperparameter grid, and metric
gsgbr = GridSearchCV(my_pipe , hyperparam_grid ,  scoring = 'r2')

# Fit GridSearchCV on the entire dataset (xs, ys)
gsgbr.fit(xs,ys)
             
# Print the best R-squared score from GridSearchCV
gsgbr.best_score_
print(gsgbr.best_score_)










-0.015043012741787588


In [3]:
from sklearn.metrics import r2_score, mean_squared_error

predictions = gsgbr.predict(test_x)
# Print the R-squared score on the test data
r2 = r2_score(test_y, predictions)
print("R-squared score on test data:", r2)

# Print the Mean Squared Error on the test data
mse = mean_squared_error(test_y, predictions)
print("Mean Squared Error on test data:", mse)

# Print the first few predictions for comparison
print("Sample predictions:", predictions[:5])
print("Actual values for comparison:", test_y.values[:5])

# Print the best score found by GridSearchCV
print("Best score from GridSearchCV:", gsgbr.best_score_)
print("Best parameters found by GridSearchCV:", gsgbr.best_params_)

# Test predictions on the test data
predictions = gsgbr.predict(test_x)
print("Sample predictions:", predictions[:5])
print("Actual values for comparison:", test_y.values[:5])

R-squared score on test data: 0.24951527375153648
Mean Squared Error on test data: 2.2500014519793037e-05
Sample predictions: [ 8.97566819e-05  1.67649860e-04 -5.26453249e-05  1.64313810e-03
  1.12662574e-04]
Actual values for comparison: [0.01201665 0.00029082 0.00018791 0.00733264 0.00054073]
Best score from GridSearchCV: -0.015675288769604334
Best parameters found by GridSearchCV: {'Gradient_Boosting_Regressor__learning_rate': 0.01, 'Gradient_Boosting_Regressor__max_depth': 10, 'Gradient_Boosting_Regressor__n_estimators': 100}
Sample predictions: [ 8.97566819e-05  1.67649860e-04 -5.26453249e-05  1.64313810e-03
  1.12662574e-04]
Actual values for comparison: [0.01201665 0.00029082 0.00018791 0.00733264 0.00054073]
